# Bio-3D｜免費 Colab 建模實驗 v1.0

**目標：** 從拜爾原始角色設定圖的「正面角色圖」取得真正的 3D 網格（GLB），用來評估 AI 是否能重建拜爾的形體。

這份 Notebook 使用開源 [TripoSR](https://github.com/VAST-AI-Research/TripoSR)；**一次只重建一張圖片**，並非四視角融合。請優先上傳從原圖裁切的 **front.png**，不是整張設定圖。原圖的側面和背面留待 Blender 校正。

**執行：** Colab 選單「執行階段 → 變更執行階段類型 → T4 GPU（若有）」；然後自上而下依序執行。免費 GPU 不保證供應；安裝或執行失敗時請保留最後錯誤訊息。

In [ ]:
import sys, os, shutil, subprocess, torch
print("Python:",sys.version.split()[0])
print("PyTorch:",torch.__version__)
print("GPU available:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
    print("VRAM GiB:",round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
else:
    raise RuntimeError("目前沒有可用 GPU。請在「執行階段 → 變更執行階段類型」選擇 GPU，若免費 GPU 額度已用完，請稍後重試。")
print("Free disk GiB:",round(shutil.disk_usage('/content').free/2**30,2))

## 1. 安裝官方 TripoSR 專案及套件

第一次需要下載開源程式與模型權重。官方 requirements 含部分舊版套件，Colab Python 版本可能需要額外相容性修正；**如果安裝失敗，停止往下跑，保存錯誤給我們**。

In [ ]:
%cd /content
!test -d TripoSR || git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git
%cd /content/TripoSR
# Colab 預裝 torch，避免 requirements 意外更換 CUDA/PyTorch。
!python -m pip install -q --upgrade pip setuptools wheel ninja
!python -m pip install -q -r requirements.txt
print("依賴安裝階段已結束。")

## 2. 上傳拜爾正面 PNG

建議使用 [Bio-3D 參考圖套件](https://github.com/hwh0410/Bio-3D/blob/main/docs/NEW_MODEL_PIPELINE.md) 裡的 **front.png**。TripoSR 通常對「單一主體、乾淨背景」效果較好。上傳檔案只存在這次 Colab 執行階段，關閉或重置後可能消失。

In [ ]:
from google.colab import files
from PIL import Image
from pathlib import Path
uploaded=files.upload()
allowed={'.png','.jpg','.jpeg','.webp'}
names=[n for n in uploaded if Path(n).suffix.lower() in allowed]
assert names,"請上傳 PNG/JPG/WEBP 格式的拜爾正面圖。"
src=Path(names[0])
out=Path('/content/TripoSR/bio_front.png')
Image.open(src).convert('RGBA').save(out)
print("輸入：",out)
display(Image.open(out))

## 3. 生成真正的 3D 網格（GLB）

使用官方 CLI，預設輸出有頂點顏色的 GLB。若遇 CUDA out of memory，可把 `--mc-resolution 192` 改成 `128`，或把 `--chunk-size 4096` 改成 `2048`。這只是初稿，不代表符合拜爾的設計圖。

In [ ]:
%cd /content/TripoSR
!python run.py bio_front.png --output-dir /content/Bio-TripoSR-output --model-save-format glb --mc-resolution 192 --chunk-size 4096
from pathlib import Path
glb=Path("/content/Bio-TripoSR-output/0/mesh.glb")
assert glb.is_file() and glb.stat().st_size>1000, "尚未產生有效 GLB，請檢查上一格錯誤紀錄。"
print("GLB 生成完成：",glb,"大小 MiB",round(glb.stat().st_size/2**20,2))

## 4. 確認網格並下載

檢查是否真的有面與頂點，避免誤認為空白模型。完成後下載 GLB；你可以在 [Bio Master Viewer](https://hwh0410.github.io/Bio-3D/master-preview.html) 選擇本機檔案預覽、旋轉、截透明 PNG。**不要在驗收前直接覆蓋 repo 正式模型。**

In [ ]:
import trimesh
asset=trimesh.load(str(glb),force='scene')
meshes=[g for g in asset.geometry.values() if isinstance(g,trimesh.Trimesh)]
assert meshes and sum(len(g.vertices) for g in meshes)>0,"GLB 內沒有有效 mesh"
print("網格數量:",len(meshes))
print("總頂點數:",sum(len(g.vertices) for g in meshes))
print("總面數:",sum(len(g.faces) for g in meshes))
from google.colab import files
files.download(str(glb))

## 後續流程

1. 先在 GitHub Pages 的 **Master Viewer** 以正面、3/4、側面、背面檢視初稿。
2. 對比原圖：安全帽寬度、頭殼輪廓、臉部面板、胸甲、手套、靴子。
3. 如果正面輪廓尚有明顯差距，先不要花時間綁骨。
4. Blender 修正模型外型後再輸出 `models/Bio_Master.glb`，作為可持續迭代的正式母版。

**技術限制：** TripoSR 是單圖重建，無法直接忠實復原未出現在照片中的背面細節。官方程式、依賴及雲端 GPU 可用性可能改動；這份 Notebook 已作靜態檢查，但尚未在 Colab GPU 上實際跑通。